# 06: Can the v1-excluded EyePACS images serve as a fresh confirmation set?

v1 used 19,332 of the 35,126 labelled EyePACS images. The recovered v1 setup notebook shows
the exclusion was not a quality filter: labels were simply restricted to whichever images had
been preprocessed. This notebook checks, **without running any model or evaluation**:

1. **Patient independence.** The excluded images' patients must share nothing with the
   9,666 patients in our train / val / test splits.
2. **Availability.** Every excluded image must exist in the raw archive
   (`data/eyepacs/raw/train_combined.zip`, 35 GB). Also reports whether the processed batches
   happen to contain any excluded images.
3. **Preprocessing fidelity.** Re-runs the recovered v1 pipeline (RGB, 512x512 Lanczos, PNG)
   on raw copies of *already-processed* images and compares pixel by pixel. If they match, the
   excluded images can be preprocessed identically to what the models were trained on.
4. **Composition.** Grade distribution, referable rate and severe-case count, i.e. the power
   a confirmatory test would have.

Reads archives in place through the Drive mount (only the zip index and ~40 images are read),
so nothing large is downloaded. CPU runtime. Run cells 1-5.

In [5]:
# Cell 1: mount Drive; load labels and the v2 splits
from google.colab import drive
drive.mount('/content/drive')
import os, json, zipfile, io, datetime
import numpy as np, pandas as pd, PIL
from PIL import Image

BASE   = '/content/drive/MyDrive/stochastic_orthogonality/data'
RAWZIP = f'{BASE}/eyepacs/raw/train_combined.zip'
BATCHES = [f'{BASE}/eyepacs_full/eyepacs_processed_batch{i}.zip' for i in range(1, 6)]
LABELS = f'{BASE}/eyepacs_full/trainLabels.csv'
SPLITS = f'{BASE}/dr_ensemble_v2/01_splits'
AUDIT  = f'{BASE}/dr_ensemble_v2/99_audit'
for p in [RAWZIP, LABELS, SPLITS, AUDIT] + BATCHES:
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)

labels = pd.read_csv(LABELS); labels.columns = ['image_id', 'level']
labels['image_id'] = labels['image_id'].astype(str)
used = pd.concat([pd.read_csv(f'{SPLITS}/eyepacs_split_{s}_v2.csv') for s in ['train', 'val', 'test']])
used_ids = set(used['image_id'].astype(str))
print(f'\nlabelled: {len(labels)} | used in v2 splits: {len(used_ids)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs/raw/train_combined.zip
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/trainLabels.csv
OK      /content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/01_splits
OK      /content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/99_audit
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch1.zip
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch2.zip
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch3.zip
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch4.zip
OK      /content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch5.zip

la

In [6]:
# Cell 2: the excluded set: size, patient independence, composition
ex = labels[~labels['image_id'].isin(used_ids)].copy()
ex['patient_id'] = ex['image_id'].str.split('_').str[0]
ex['eye'] = ex['image_id'].str.split('_').str[-1]
used_patients = set(used['patient_id'].astype(str))
ex_patients = set(ex['patient_id'])
overlap = ex_patients & used_patients
eyes_per_patient = ex.groupby('patient_id')['eye'].nunique()

COMP = {
  'excluded_images': int(len(ex)), 'excluded_patients': int(len(ex_patients)),
  'patient_overlap_with_v2_splits': int(len(overlap)),
  'patients_with_both_eyes': int((eyes_per_patient == 2).sum()),
  'patients_with_one_eye': int((eyes_per_patient == 1).sum()),
  'grade_dist': {int(a): int(b) for a, b in ex['level'].value_counts().sort_index().items()},
  'referable_rate': round(float((ex['level'] >= 2).mean()), 4),
  'severe_grade3plus': int((ex['level'] >= 3).sum()), 'grade4': int((ex['level'] == 4).sum()),
}
print(json.dumps(COMP, indent=2))
print('\nfor comparison, the spent v2 test set had 2,900 images, 130 severe (55 grade 4)')
print('PATIENT-INDEPENDENT' if not overlap else f'WARNING: {len(overlap)} shared patients')

{
  "excluded_images": 15794,
  "excluded_patients": 7897,
  "patient_overlap_with_v2_splits": 0,
  "patients_with_both_eyes": 7897,
  "patients_with_one_eye": 0,
  "grade_dist": {
    "0": 11578,
    "1": 1131,
    "2": 2373,
    "3": 384,
    "4": 328
  },
  "referable_rate": 0.1953,
  "severe_grade3plus": 712,
  "grade4": 328
}

for comparison, the spent v2 test set had 2,900 images, 130 severe (55 grade 4)
PATIENT-INDEPENDENT


In [7]:
# Cell 3: availability in the raw archive and in the processed batches
def index_zip(path):
    with zipfile.ZipFile(path) as zf:
        return {os.path.splitext(os.path.basename(n))[0]: n for n in zf.namelist() if not n.endswith('/')}

RAW_IDX = index_zip(RAWZIP)
print(f'raw archive entries: {len(RAW_IDX)} | sample names: {list(RAW_IDX.values())[:3]}')
PROC_IDX = {}
for b in BATCHES:
    for k, v in index_zip(b).items(): PROC_IDX[k] = (b, v)
print(f'processed batch entries: {len(PROC_IDX)} | sample: {list(PROC_IDX.values())[:2]}')

AVAIL = {
  'raw_archive_entries': len(RAW_IDX),
  'excluded_in_raw': int(ex['image_id'].isin(RAW_IDX.keys()).sum()),
  'excluded_missing_from_raw': int((~ex['image_id'].isin(RAW_IDX.keys())).sum()),
  'processed_batch_entries': len(PROC_IDX),
  'excluded_already_in_processed_batches': int(ex['image_id'].isin(PROC_IDX.keys()).sum()),
  'used_in_processed_batches': int(sum(i in PROC_IDX for i in used_ids)),
}
print(json.dumps(AVAIL, indent=2))

raw archive entries: 1 | sample names: ['train.zip.005']
processed batch entries: 19332 | sample: [('/content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch1.zip', '2831_left.png'), ('/content/drive/MyDrive/stochastic_orthogonality/data/eyepacs_full/eyepacs_processed_batch1.zip', '14129_right.png')]
{
  "raw_archive_entries": 1,
  "excluded_in_raw": 0,
  "excluded_missing_from_raw": 15794,
  "processed_batch_entries": 19332,
  "excluded_already_in_processed_batches": 0,
  "used_in_processed_batches": 19332
}


In [8]:
# Cell 4: preprocessing fidelity: recovered v1 pipeline on raw vs the processed versions the models saw
def v1_preprocess(fp):
    """Recovered from v1 notebook1_data_setup: RGB, resize 512x512 Lanczos (aspect not preserved), PNG."""
    return Image.open(fp).convert('RGB').resize((512, 512), Image.LANCZOS)

rng = np.random.default_rng(20260922)
used_df = used[['image_id', 'level']].copy(); used_df['image_id'] = used_df['image_id'].astype(str)
sample = pd.concat([g.sample(min(8, len(g)), random_state=int(rng.integers(1e9)))
                    for _, g in used_df.groupby('level')])
rows = []
zraw = zipfile.ZipFile(RAWZIP); zproc = {b: zipfile.ZipFile(b) for b in BATCHES}
for iid in sample['image_id']:
    if iid not in RAW_IDX or iid not in PROC_IDX: continue
    rep = v1_preprocess(io.BytesIO(zraw.read(RAW_IDX[iid])))
    b, name = PROC_IDX[iid]
    proc = Image.open(io.BytesIO(zproc[b].read(name)))
    pa = np.asarray(proc.convert('RGB'), dtype=np.int16); ra = np.asarray(rep, dtype=np.int16)
    same_shape = pa.shape == ra.shape
    d = np.abs(pa - ra) if same_shape else None
    rows.append({'image_id': iid, 'proc_size': proc.size, 'proc_mode': proc.mode, 'proc_format': proc.format,
                 'same_shape': same_shape,
                 'max_abs_diff': int(d.max()) if same_shape else None,
                 'mean_abs_diff': round(float(d.mean()), 4) if same_shape else None,
                 'identical': bool(same_shape and d.max() == 0)})
FID = pd.DataFrame(rows)
print(FID.to_string(index=False))
print(f"\nPIL {PIL.__version__} | compared {len(FID)} | identical {int(FID['identical'].sum())} | "
      f"worst mean abs diff {FID['mean_abs_diff'].max()} | worst max abs diff {FID['max_abs_diff'].max()}")

Empty DataFrame
Columns: []
Index: []


KeyError: 'identical'

In [ ]:
# Cell 5: verdict and record
fid_ok = bool(len(FID) > 0 and FID['same_shape'].all() and FID['mean_abs_diff'].max() < 1.0)
VERDICT = {
  'patient_independent': COMP['patient_overlap_with_v2_splits'] == 0,
  'all_excluded_available_raw': AVAIL['excluded_missing_from_raw'] == 0,
  'preprocessing_reproduces_v1': fid_ok,
  'usable_as_confirmation_set': bool(COMP['patient_overlap_with_v2_splits'] == 0 and
                                     AVAIL['excluded_missing_from_raw'] == 0 and fid_ok),
}
REC = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'pil_version': PIL.__version__,
       'composition': COMP, 'availability': AVAIL, 'verdict': VERDICT,
       'fidelity': FID.astype(str).to_dict(orient='records'),
       'note': 'Label-free except aggregate composition; no model was run and no evaluation performed.'}
with open(f'{AUDIT}/fresh_set_check_v2.json', 'w') as f: json.dump(REC, f, indent=2, default=str)
print(json.dumps(VERDICT, indent=2))
print(f'\nwrote {AUDIT}/fresh_set_check_v2.json')

In [9]:
import struct
p = RAWZIP; size = os.path.getsize(p)
print(f'size: {size/1e9:.2f} GB')
with open(p, 'rb') as f:
    head = f.read(512)
    f.seek(max(0, size - (1 << 20))); tail = f.read()
print('starts with zip header:', head[:4] == b'PK\x03\x04')
if head[:4] == b'PK\x03\x04':
    n = struct.unpack('<H', head[26:28])[0]
    print('first entry name:', head[30:30 + n])
print('end-of-archive records in last 1 MB:', tail.count(b'PK\x05\x06'), '| zip64:', tail.count(b'PK\x06\x06'))
with zipfile.ZipFile(p) as zf:
    for zi in zf.infolist():
        print(f'{zi.filename}  {zi.file_size/1e9:.2f} GB  offset {zi.header_offset/1e9:.2f} GB')

size: 35.00 GB
starts with zip header: True
first entry name: b'train.zip.001'
end-of-archive records in last 1 MB: 1 | zip64: 0
train.zip.005  1.43 GB  offset 33.56 GB
